# FE 5108 — Stage 1: Portfolio choice and estimation error

This notebook is the **single reproducible entry point** for the project: run it
top to bottom (*Kernel ▸ Restart & Run All*) and it regenerates every Stage 1
exhibit from the committed raw CSVs in `data/`.

All of the logic lives in the `goldman` package (`src/goldman/`); the notebook is
a thin, readable driver over the modules so you can see exactly what each step
does. Stage 1 asks for:

- **(1a)** summary statistics of the universe,
- **(1b)** the tangency weights next to the value weights, as one chart,
- **(1c)** a discussion of estimation error, with a subsample re-estimation.

**Setup.** `uv sync --extra notebook`, then open this file. The internet is only
needed to refresh data via `python -m goldman.data.fetch`; the committed CSVs are
enough to run the exhibits.


## Setup
Import the data and analysis modules and pick the output directory.

In [1]:
from goldman.analysis import (
    plot_weights,
    run_portfolio_analysis,
    subsample_tangency,
    summary_statistics,
    tangency_vs_comparison,
    weight_correlation,
)
from goldman.config import RF_MONTHLY, ensure_outputs_dir
from goldman.data import load_ff3, load_returns, load_value_weights, monthly_rf

UNIVERSE = "us"          # "us" or "sg"; "synthetic" is fake smoke-test data
OUTPUT_DIR = ensure_outputs_dir()
print("outputs ->", OUTPUT_DIR.name)

outputs -> outputs


## Load and clean the return panel

All cleaning decisions (sparse-ticker drops, fully-observed months) live once in
`goldman.data`, so this notebook and the tests see the same panel.

In [2]:
returns = load_returns(universe=UNIVERSE)
print(
    f"{returns.shape[1]} assets x {returns.shape[0]} months "
    f"({returns.index[0]:%Y-%m} .. {returns.index[-1]:%Y-%m})"
)
returns.iloc[:3, :5].round(4)

30 assets x 119 months (2016-10 .. 2026-08)


,AAPL,AMZN,BAC,CSCO,CVX
Date,,,,,
2016-10-01,0.0043,-0.0567,0.0543,-0.0248,0.0178
2016-11-01,-0.0216,-0.0497,0.2847,-0.0280,0.0757
2016-12-01,0.0480,-0.0009,0.0464,0.0134,0.0550


## Risk-free rate and factor data

We use the monthly risk-free rate from the bundled Fama–French file, which is
aligned to our calendar by `monthly_rf` (falling back to a constant outside its
coverage). This is the same input the CAPM and factor stages will use later.

In [3]:
rf = monthly_rf(returns.index, ff3=load_ff3(), fallback=RF_MONTHLY)
rf.describe()

count    119.000000
mean       0.001945
std        0.001579
min        0.000000
25%        0.000350
50%        0.001800
75%        0.003400
max        0.004800
Name: RF, dtype: float64

## Point-in-time value weights

`load_value_weights` reads the long-format CSV built from per-ticker prices and
shares outstanding, aligns it to this return panel (forward-fill, so no month
ever uses a future share count) and renormalises each date. The benchmark is
therefore a genuine value-weighted index, not today's market caps applied to
old data.

In [4]:
value_weights = load_value_weights(
    universe=UNIVERSE, assets=returns.columns, dates=returns.index
)
print(f"{value_weights.shape[1]} assets x {value_weights.shape[0]} months")
value_weights.mean().sort_values(ascending=False).head(10).round(4)

30 assets x 119 months


asset
AAPL     0.1523
MSFT     0.1365
GOOGL    0.1096
AMZN     0.1014
NVDA     0.0619
META     0.0598
JPM      0.0375
JNJ      0.0350
WMT      0.0335
XOM      0.0315
dtype: float64

## (1a) Summary statistics
Annualised mean, volatility and Sharpe, plus monthly extremes.

In [5]:
summary = summary_statistics(returns, rf=rf)
summary.to_csv(OUTPUT_DIR / "summary_stats.csv")
print("wrote", "outputs/summary_stats.csv")
summary

wrote outputs/summary_stats.csv


,mean (ann.),vol (ann.),sharpe (ann.),min month,max month
AAPL,0.292,0.272,0.985,-0.181,0.217
AMZN,0.233,0.312,0.670,-0.238,0.273
BAC,0.202,0.286,0.626,-0.250,0.285
CSCO,0.186,0.249,0.655,-0.155,0.316
CVX,0.153,0.285,0.453,-0.224,0.273
DIS,0.068,0.306,0.146,-0.186,0.249
GOOGL,0.253,0.270,0.852,-0.179,0.338
HD,0.145,0.229,0.530,-0.151,0.189
INTC,0.224,0.532,0.377,-0.354,1.141
JNJ,0.124,0.173,0.585,-0.122,0.144


## (1b) Tangency weights vs value weights

The estimated tangency portfolio is `w_F ∝ Σ̂⁻¹ μ̂_e`; the comparison is the
value-weighted index. The chart uses the sample-average value-weight vector (the
full monthly panel is written out by `run_portfolio_analysis` below).

In [6]:
comparison_label = "value weights (sample average)"
weight_table = tangency_vs_comparison(returns, rf=rf, comparison=value_weights)
print(f"most extreme tangency weight: {weight_table['tangency'].abs().max():+.0%}")
weight_table.head()

most extreme tangency weight: +280%


,tangency,comparison
AAPL,0.449211,0.152336
AMZN,0.403924,0.101441
BAC,-0.466926,0.025369
CSCO,0.297125,0.018440
CVX,0.563913,0.021576


In [7]:
plot_weights(
    weight_table,
    OUTPUT_DIR / "tangency_weights.png",
    comparison_label=comparison_label,
)
print("wrote", "outputs/tangency_weights.png")

wrote outputs/tangency_weights.png


**Takeaway (1b).** The tangency portfolio takes large long–short positions
(>300% in a single name) while the value benchmark holds small, positive
weights. The two disagree because the tangency weights invert a noisy `Σ̂` and
amplify noisy `μ̂` — the Monte-Carlo of estimation error, not a real signal.

## (1c) Estimation error: subsample re-estimation
Re-estimate on each half of the sample and see how far the weights move.

In [8]:
drift = subsample_tangency(returns, rf=rf).round(3)
drift.to_csv(OUTPUT_DIR / "subsample_weights.csv")
corr = weight_correlation(drift, "first half", "second half")
print(f"correlation between first- and second-half weight vectors: {corr:.2f}")
drift.head()

correlation between first- and second-half weight vectors: 0.35


,first half,second half,full sample
AAPL,0.127,1.346,0.449
AMZN,-0.043,0.111,0.404
BAC,-0.110,-2.978,-0.467
CSCO,0.285,0.508,0.297
CVX,0.572,-0.071,0.564


**Takeaway (1c).** A near-zero (or negative) correlation between the two
half-sample weight vectors says the estimates are unstable: the same theory,
the same assets, a different window — very different portfolios. This is why
naive mean–variance is hard to deploy and why shrinkage or constraints help.

## One-call reproduction

`run_portfolio_analysis` runs all of the above and writes every exhibit,
including the full monthly value-weight panel (`value_weights.csv`).

In [9]:
result = run_portfolio_analysis(
    returns,
    rf=rf,
    comparison=value_weights,
    comparison_label=comparison_label,
    output_dir=OUTPUT_DIR,
)
for name, path in result.output_paths.items():
    print(f"{name:14s} outputs/{path.name}")

summary        outputs/summary_stats.csv
weights        outputs/tangency_weights.png
subsample      outputs/subsample_weights.csv
value_weights  outputs/value_weights.csv


## Deliverables and caveats

**Outputs written to `outputs/`**

| Exhibit | File |
|---|---|
| (1a) summary statistics | `summary_stats.csv` |
| (1b) tangency vs value weights | `tangency_weights.png`, `value_weights.csv` |
| (1c) subsample drift | `subsample_weights.csv` |

**Data sources.** Yahoo adjusted closes (total-return proxy); Ken French FF3 for
the risk-free rate; market caps built from point-in-time shares outstanding.

**Honest caveats to carry into the report**

- Both universes are *today's members backfilled* — survivorship bias, and it
  flatters average returns and the SML.
- Value weights are point-in-time, but a few names (META before 2022-07, some SG
  listings) lack early Yahoo share history and carry their earliest known count
  backward.
- The risk-free proxy is the FF3 one-month T-bill; state this once.

Stages 2–4 (the CAPM/SML, factor models, and the committee memo) slot into
`src/goldman/analysis/` and get their own notebook sections as they are built.
